# Tabula Sapiens: the CELLxGENE UMAP and cell-type signatures in Celldega

**Dev draft; not part of the documentation.** CELLxGENE Discover's homepage sends
you to [*Tabula Sapiens - All Cells*](https://cellxgene.cziscience.com/e/53d208b0-2cfd-4366-9866-c3c6114081bc.cxg/)
in its Explorer: 1,136,218 cells from 28 organs of 24 donors, on the authors'
UMAP. This notebook shows the same view in Celldega, the UMAP colored by cell
type with the Explorer's colors, and adds what the Explorer lacks: the 180 cell
types' expression signatures in a Clustergram, linked to the UMAP and to Enrich,
with a slider for the number of markers per cell type.

Like a baking show, the recipe ran once: `Tabula_Sapiens_pre-process.ipynb`
streamed the 45 GB source into the stores below, which this notebook loads from
the [`Tabula_Sapiens/` folder](https://huggingface.co/datasets/broadinstitute/Celldega_Supporting_Data/tree/main/Tabula_Sapiens)
of the Celldega Supporting Data on Hugging Face, or from the local copies the
recipe writes to the Git-ignored `notebooks/data/tabula_sapiens/baked/`:

- `tabula_sapiens.zarr` (AnnData Zarr): cells, annotations, colors, the UMAP, and
  a column-compressed copy of the log1p expression, read by Landscape in the browser one
  gene at a time.
- `tabula_sapiens_cell_type_markers.zarr` (`Matrix` Zarr): mean expression and
  fraction of expressing cells for every cell type, computed from all 1.1M cells,
  with t-test marker ranks and the clustering already done.
- `tabula_sapiens_cell_type_signatures.zarr` (AnnData Zarr): every cell type's
  mean expression of every gene, for the Landscape's viewport estimates.
- `tabula_sapiens_plasma_cells.zarr` and `tabula_sapiens_plasma_cell_markers.zarr`:
  the same pair for the 27,349 plasma cells, re-clustered into Leiden sub-clusters.

This needs Celldega 0.28.0a2 or later (`pip install --pre celldega`, or this
checkout), `zarr>=3`, and `anndata>=0.12,<0.13`. Run it in JupyterLab from the
`notebooks/` directory.

**Data and attribution.** [Tabula Sapiens - All Cells](https://cellxgene.cziscience.com/e/53d208b0-2cfd-4366-9866-c3c6114081bc.cxg/)
(Tabula Sapiens 2.0; dataset version `1c88f927-bcbb-4bb1-9881-281842945a2d`),
curated and distributed by CZ CELLxGENE Discover in the
[Tabula Sapiens collection](https://cellxgene.cziscience.com/collections/e5f58829-1a66-40b5-a624-9046778e74f5)
under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

- The Tabula Sapiens Consortium. Tabula Sapiens reveals transcription factor
  expression, senescence effects, and sex-specific features in cell types from 28
  human organs and tissues. *bioRxiv* (2024).
  [doi:10.1101/2024.12.03.626516](https://doi.org/10.1101/2024.12.03.626516)
  (the citation the [Tabula Sapiens data release policy](https://tabula-sapiens.sf.czbiohub.org/whereisthedata)
  asks for with the 2.0 data; apparently published as *Cell* 189, 7212–7227 (2026),
  [doi:10.1016/j.cell.2026.08.010](https://doi.org/10.1016/j.cell.2026.08.010)).
- The Tabula Sapiens Consortium. The Tabula Sapiens: A multiple-organ,
  single-cell transcriptomic atlas of humans. *Science* 376, eabl4896 (2022).
  [doi:10.1126/science.abl4896](https://doi.org/10.1126/science.abl4896)

The stores are derived from that data: the expression is normalized to 10,000
counts per cell and log1p-transformed, and the cell-type signatures, marker ranks,
and plasma-cell sub-clusters are our analyses, not the Tabula Sapiens
Consortium's. The pre-process notebook lists every change; the Hugging Face
folder's README adds the license and the full text of the Tabula Sapiens data
release policy, which applies to any redistribution.

## Setup

Celldega's front end comes from the published npm alpha, `celldega@0.28.0-alpha.2`,
which has the UMAP/XY viewport bars and the UMAP orientation; each widget saves
only a small loader for it. To try a local build (`npm run build`), set
`CELLDEGA_LOCAL_ESM=1` instead: that inlines the ~12 MB bundle into every widget,
so clear the outputs before committing.

In [ ]:
import os

# Set before importing Celldega.
os.environ.setdefault("CELLDEGA_ESM_VERSION", "0.28.0-alpha.2")

from pathlib import Path

import celldega as dega

# Hugging Face serves each file at its `resolve` URL, like GitHub's raw URLs. The
# local copies the pre-process notebook writes are used when they exist.
HF_URL = "https://huggingface.co/datasets/broadinstitute/Celldega_Supporting_Data/resolve/main/Tabula_Sapiens"

cwd = Path.cwd().resolve()
project_root = next(
    (p for p in (cwd, *cwd.parents) if (p / "pyproject.toml").is_file() and (p / "notebooks").is_dir()),
    None,
)
baked_dir = (project_root / "notebooks" if project_root else cwd) / "data" / "tabula_sapiens" / "baked"


def baked(name):
    local = baked_dir / name
    return local if local.exists() else f"{HF_URL}/{name}"


zarr_url = baked("tabula_sapiens.zarr")
matrix_url = baked("tabula_sapiens_cell_type_markers.zarr")
signature_url = baked("tabula_sapiens_cell_type_signatures.zarr")
plasma_zarr_url = baked("tabula_sapiens_plasma_cells.zarr")
plasma_matrix_url = baked("tabula_sapiens_plasma_cell_markers.zarr")

# The cell annotations Landscape reads; the store has more (see the last section).
CELL_ATTR = ["cell_type", "tissue_in_publication", "compartment"]

## 1. The atlas, as CELLxGENE shows it

The authors' `X_umap`, colored by `cell_type`. CELLxGENE Explorer has no author
palette for cell types, so it spreads d3's rainbow over them in their stored
order; the store holds the same colors. The UMAP's second coordinate increases
upward, as in CELLxGENE and Scanpy. Landscape loads the cells, cell types, and
UMAP when it opens, then fetches a gene's expression when you pick one in the gene
bar.

**Zoom in** and the bars follow the viewport, as they do in a spatial dataset:

- The cell-type bars count the cells in view, and points grow as you zoom.
- The gene bars estimate expression in view by weighting each cell type's mean
  expression by its cells in view; **TRX turns purple** to mark the estimate.
  It is blue when every cell is in view and the bars show the whole dataset.
  The means come from the pre-process notebook's signature store
  (`gene_signature_url`): the estimate reads its `log1p` layer, the same
  library-size-normalized, log1p values that color the cells. Its `counts`
  layer is kept for reference only, since Smart-seq2 read counts dwarf 10x UMIs.
- **Click TRX** for the exact mean expression of the top 50 estimated genes in the
  cells in view (TRX turns blue); panning returns to the estimate.

To color by organ or compartment instead, pass `cluster_attr="tissue_in_publication"`
or `"compartment"`: the store holds the authors' palettes for both, as the Explorer
uses them.

In [ ]:
landscape = dega.viz.Landscape(
    zarr_url=zarr_url,
    zarr_layer="log1p_csc",
    gene_signature_url=signature_url,
    cell_attr=CELL_ATTR,
    cluster_attr="cell_type",
    dataset_name="Tabula Sapiens · All Cells",
    height=750,
    ini_zoom=-4.3,
    ini_x=5_000,
    ini_y=5_000,
)
landscape

## 2. Cell-type signatures, linked to the UMAP and Enrich

Each Clustergram column is a cell type and each row a gene: dot color is the
gene's Z-scored mean log-normalized expression, dot size the fraction of the cell
type's cells expressing it. The column bars show each cell type's color, its
majority compartment, and its number of cells (`n_cells`): double-click an
attribute's label to order the columns by it, e.g. by cell count.

The **RANK** slider sets how many top markers per cell type are shown, from 1 to
50 (at most 9,000 genes for 180 cell types) or **all**, each cell type's top 100
together. Markers are ranked by Scanpy's `t-test`: each cell type against all
other cells, computed exactly on the full dataset.

A second Landscape of the same store sits next to the Clustergram, so each view
has its own UMAP.

In [ ]:
mat = dega.clust.Matrix.read_zarr(matrix_url)
cgm = dega.viz.Clustergram(matrix=mat, width=800, height=550, viz_mode="dotplot", rank_dim=3)

landscape_linked = dega.viz.Landscape(
    zarr_url=zarr_url,
    zarr_layer="log1p_csc",
    gene_signature_url=signature_url,
    cell_attr=CELL_ATTR,
    cluster_attr="cell_type",
    dataset_name="Tabula Sapiens · All Cells",
    height=750,
    ini_zoom=-4.6,
    ini_x=5_000,
    ini_y=5_000,
)
linked = dega.viz.spatial_clustergram(
    landscape_linked,
    cgm,
    width="600px",
    height="750px",
    enrich=True,
    enrich_kwargs={"inst_lib": "CellMarker_2024"},
)

In [ ]:
linked

Try these interactions:

- Click a **cell-type column** to highlight its cells in the UMAP.
- Click a **gene row**, e.g. `PTPRC`, `EPCAM`, or `PECAM1`, to color the cells
  by its expression.
- Select a **gene dendrogram** group or crop rows to send the genes to Enrich
  (CellMarker 2024), and click a gene in Enrich to color the cells by it.
- Move the **RANK** slider for more or fewer markers per cell type.
- Double-click the **n_cells** label to order the cell types by size.

## 3. Plasma-cell sub-clusters

The 27,349 plasma cells, re-clustered on their own: a new UMAP and Leiden
sub-clusters from the authors' batch-corrected scVI embedding, so cells group by
biology rather than by donor or assay. Each Clustergram column is a sub-cluster,
with its majority organ and cell count; markers are Scanpy's Wilcoxon test of
each sub-cluster against the other plasma cells.

Things to look for: antibody isotype (`IGHA1`/`IGHA2` with `JCHAIN` in the gut
against `IGHG1`–`IGHG4` in lymph node and spleen), proliferating plasmablasts
(`MKI67`), and sub-clusters marked by stress (`FOS`, `JUN`) or mitochondrial
genes, which are likely technical rather than new cell states.

Here the viewport signatures are computed on the fly: load the plasma cells'
log1p expression (27,349 cells), average it per sub-cluster with Scanpy, and pass
the result as `adata_sig`, sent with the widget like `adata`.

In [ ]:
import anndata as ad
import scanpy as sc

plasma_cells = ad.read_zarr(plasma_zarr_url)
plasma_sig = sc.get.aggregate(plasma_cells, by="leiden", func="mean", layer="log1p_csc")
plasma_sig.layers["log1p"] = plasma_sig.layers.pop("mean")
plasma_sig.uns["viewport_signature"] = {
    "groupby": "leiden",
    "layer": "log1p",
    "units": "mean log1p counts per 10,000",
}
plasma_sig

In [ ]:
plasma_landscape = dega.viz.Landscape(
    zarr_url=plasma_zarr_url,
    zarr_layer="log1p_csc",
    adata_sig=plasma_sig,
    cell_attr=["leiden", "tissue_in_publication"],
    cluster_attr="leiden",
    dataset_name="Tabula Sapiens · plasma cells",
    height=750,
    ini_zoom=-4.6,
    ini_x=5_000,
    ini_y=5_000,
)
plasma_cgm = dega.viz.Clustergram(
    matrix=dega.clust.Matrix.read_zarr(plasma_matrix_url),
    width=470,
    height=550,
    viz_mode="dotplot",
    rank_dim=5,
)
plasma_linked = dega.viz.spatial_clustergram(
    plasma_landscape,
    plasma_cgm,
    width="600px",
    height="750px",
    enrich=True,
    enrich_kwargs={"inst_lib": "CellMarker_2024"},
)

In [ ]:
plasma_linked

## Selected cells

With a running kernel, `get_selected_cells()` returns the cell IDs of the selected
cell type, Clustergram selection, or drawn gate. The store's expression takes several GB, so
read only its `obs` table (about 1.1M rows) to look the cells up. The plasma-cell
view uses the same cell IDs, so `plasma_landscape.get_selected_cells()` works here
too.

In [ ]:
import zarr

obs = ad.io.read_elem(zarr.open_group(zarr_url, mode="r")["obs"])
selected = landscape_linked.get_selected_cells()
print(f"{len(selected):,} cells selected")
obs.loc[selected, CELL_ATTR].value_counts().head(10)

## Next

- **Single-cell Clustergram.** About 5,000 sampled plasma cells (or cells of one
  organ) as columns, to see marker variation within each sub-cluster.
- **UMAP hextiles.** Tile the UMAP with hexagons and aggregate each tile's cells into
  a signature colored by its majority cell type, for a Clustergram finer than cell
  types but far smaller than 1.1M cells. Tiling each cell type separately would show
  variation within it.